# Mid-circuit measurement with DSP classification

This notebook runs a single-qubit mid-circuit measurement experiment with `Experiment.execute(...)`. Each repetition is a single-shot experiment (`mode="single"`), and we repeat it many times to confirm that the first and second readout outcomes cluster on `gg` and `ee`.

The DSP classifier is enabled with `classification_source="gmm_linear"`, which derives the hardware decision line from the stored GMM state centers.


In [1]:
import matplotlib.pyplot as plt
import numpy as np

import qubex as qx


In [2]:
exp = qx.Experiment(
    chip_id="64Qv3",
    muxes=[6],
)
exp.connect()

target = exp.qubit_labels[2]
readout_target = exp.ctx.resolve_read_label(target)
target, readout_target


date: 2026-03-26 16:56:13
python: 3.10.18
qubex: 1.5.0b4
env: /home/sumida/workspace/development/dsp_classification/260323_qubex15/test_venv
config: /home/shared/qubex-config/64Qv3/config
params: /home/shared/qubex-config/64Qv3/params
chip: 64Qv3 (2023-1st-64Q-No14-run3 chip (1,0))
qubits: ['Q24', 'Q25', 'Q26', 'Q27']
muxes: ['MUX06']
boxes: ['S159A']
Successfully connected.


('Q26', 'RQ26')

In [ ]:
#exp.configure()


You are going to configure the following boxes:

S159A (QuEL-1 SE (7-11GHz))

This operation will overwrite the existing backend settings. Do you want to continue?
 [y/n]:

Please enter Y or N

You are going to configure the following boxes:

S159A (QuEL-1 SE (7-11GHz))

This operation will overwrite the existing backend settings. Do you want to continue?
 [y/n]:

In [4]:
exp.tool.print_target_frequencies(exp.qubits)

                                        TARGET FREQUENCIES                                        
┏━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ LABEL   ┃    LO ┃      NCO ┃     CNCO ┃     FNCO ┃    F_FINE ┃  F_TARGET ┃   F_DIFF ┃    F_AWG ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ MUX06   │  8500 │ 1500.000 │ 1500.000 │   +0.000 │ 10000.000 │ 10000.000 │   +0.000 │   +0.000 │
├─────────┼───────┼──────────┼──────────┼──────────┼───────────┼───────────┼──────────┼──────────┤
│ Q24     │ 10500 │ 2484.375 │ 2085.938 │ +398.438 │  8015.625 │  8016.067 │   +0.442 │   -0.442 │
│ Q24-Q10 │ 10500 │ 1851.562 │ 2085.938 │ -234.375 │  8648.438 │  8642.569 │   -5.869 │   +5.869 │
│ Q24-Q21 │ 10500 │ 1687.500 │ 2085.938 │ -398.438 │  8812.500 │  8757.094 │  -55.406 │  +55.406 │
│ Q24-Q25 │ 10500 │ 1687.500 │ 2085.938 │ -398.438 │  8812.500 │  8846.136 │  +33.636 │  -33.636 │
│ Q24-Q26 │ 10500 │ 1687.500 │ 2085.938 │ -398.438 │  8812.500 │  8833.222 │  +20.722 │  -20.722 │
│ RQ24    │  9000 │ 1335.938 │ 1335.938 │   +0.000 │ 10335.938 │ 10226.600 │ -109.338 │ -109.338 │
├─────────┼───────┼──────────┼──────────┼──────────┼───────────┼───────────┼──────────┼──────────┤
│ Q25     │ 10500 │ 1664.062 │ 2132.812 │ -468.750 │  8835.938 │  8846.136 │  +10.198 │  -10.198 │
│ Q25-Q11 │ 10500 │ 2625.000 │ 2132.812 │ +492.188 │  7875.000 │  7885.220 │  +10.220 │  -10.220 │
│ Q25-Q24 │ 10500 │ 2507.812 │ 2132.812 │ +375.000 │  7992.188 │  8016.067 │  +23.880 │  -23.880 │
│ Q25-Q27 │ 10500 │ 2507.812 │ 2132.812 │ +375.000 │  7992.188 │  7974.069 │  -18.119 │  +18.119 │
│ Q25-Q28 │ 10500 │ 2507.812 │ 2132.812 │ +375.000 │  7992.188 │  7973.863 │  -18.324 │  +18.324 │
│ RQ25    │  9000 │ 1335.938 │ 1335.938 │   +0.000 │ 10335.938 │ 10556.600 │ +220.662 │ +220.662 │
├─────────┼───────┼──────────┼──────────┼──────────┼───────────┼───────────┼──────────┼──────────┤
│ Q26     │ 10500 │ 1664.062 │ 2156.250 │ -492.188 │  8835.938 │  8833.222 │   -2.716 │   +2.716 │
│ Q26-Q23 │ 10500 │ 2625.000 │ 2156.250 │ +468.750 │  7875.000 │  7823.059 │  -51.941 │  +51.941 │
│ Q26-Q24 │ 10500 │ 2507.812 │ 2156.250 │ +351.562 │  7992.188 │  8016.067 │  +23.880 │  -23.880 │
│ Q26-Q27 │ 10500 │ 2507.812 │ 2156.250 │ +351.562 │  7992.188 │  7974.069 │  -18.119 │  +18.119 │
│ Q26-Q40 │ 10500 │ 2625.000 │ 2156.250 │ +468.750 │  7875.000 │  7916.718 │  +41.718 │  -41.718 │
│ RQ26    │  9000 │ 1335.938 │ 1335.938 │   +0.000 │ 10335.938 │ 10347.800 │  +11.862 │  +11.862 │
├─────────┼───────┼──────────┼──────────┼──────────┼───────────┼───────────┼──────────┼──────────┤
│ Q27     │ 10500 │ 2531.250 │ 2085.938 │ +445.312 │  7968.750 │  7974.069 │   +5.319 │   -5.319 │
│ Q27-Q25 │ 10500 │ 1664.062 │ 2085.938 │ -421.875 │  8835.938 │  8846.136 │  +10.198 │  -10.198 │
│ Q27-Q26 │ 10500 │ 1664.062 │ 2085.938 │ -421.875 │  8835.938 │  8833.222 │   -2.716 │   +2.716 │
│ Q27-Q30 │ 10500 │ 1851.562 │ 2085.938 │ -234.375 │  8648.438 │  8643.805 │   -4.633 │   +4.633 │
│ Q27-Q41 │ 10500 │ 1664.062 │ 2085.938 │ -421.875 │  8835.938 │  8847.489 │  +11.551 │  -11.551 │
│ RQ27    │  9000 │ 1335.938 │ 1335.938 │   +0.000 │ 10335.938 │ 10097.600 │ -238.338 │ -238.338 │
└─────────┴───────┴──────────┴──────────┴──────────┴───────────┴───────────┴──────────┴──────────┘

In [6]:
# Prepare the X90 pulse and store the GMM state centers used by gmm_linear.
exp.obtain_rabi_params([target], plot=True)
exp.calibrate_hpi_pulse([target], plot=True)
exp.build_classifier([target], n_states=2, n_shots=10000, plot=True)


Output()

Target: Q26
Rabi frequency: 11.9863 ± 0.1 MHz
[qubex.analysis.fitting] WARNING: R² < 0.9


Calibrating hpi pulse for Q26...



Calibration results for hpi pulse:
  Q26: 0.064708
Q26 prepared as |0⟩:


Q26 prepared as |1⟩:


Q26:
  Total shots: 10000
  |0⟩ → {0: 5571, 1: 4429}, f_0: 55.71%
  |1⟩ → {0: 5134, 1: 4866}, f_1: 48.66%
  Average readout fidelity : 52.19%




<Result created_at=2026-03-26T07:48:57.616175+00:00 data={...}>

In [7]:
wait_ns = 512
readout_duration = int(round(exp.readout_duration))
readout_pre_margin = int(round(exp.readout_pre_margin))
readout_post_margin = int(round(exp.readout_post_margin))

readout_pulse = exp.readout(
    readout_target,
    duration=readout_duration,
    pre_margin=readout_pre_margin,
    post_margin=readout_post_margin,
)

with qx.PulseSchedule([target, readout_target]) as sequence:
    sequence.add(target, exp.hpi_pulse[target])
    sequence.barrier()
    sequence.add(readout_target, readout_pulse)
    sequence.barrier()
    sequence.add(target, qx.Blank(wait_ns))
    sequence.add(readout_target, qx.Blank(wait_ns))
    sequence.barrier()
    sequence.add(readout_target, readout_pulse)

sequence


In [8]:
sequence.plot()


In [9]:
# Each repetition is a single-shot DSP-classified experiment.
n_analysis_shots = 10000
shot_interval = 150 * 1024

result = exp.execute(
    schedule=sequence,
    mode="single",
    n_shots=n_analysis_shots,
    shot_interval=shot_interval,
    enable_dsp_sum=True,
    enable_dsp_demodulation=True,
    enable_dsp_classification=True,
    classification_source="gmm_linear",
    plot=False,
)

len(result.data[target])


TypeError: ufunc 'right_shift' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
first_capture = result.data[target][0]
second_capture = result.data[target][1]

print("first raw DSP values:", np.unique(first_capture.raw))
print("second raw DSP values:", np.unique(second_capture.raw))
print("first logical labels:", np.unique(first_capture.classified))
print("second logical labels:", np.unique(second_capture.classified))

assert set(np.unique(first_capture.raw)).issubset({0, 3})
assert set(np.unique(second_capture.raw)).issubset({0, 3})

# 0 -> g, 1 -> e
shots = result.get_classified_data(targets=[(target, 0), (target, 1)])
shots[:10]


In [ ]:
bit_counts = result.get_counts(targets=[(target, 0), (target, 1)])
ordered_labels = ["00", "01", "10", "11"]
counts = np.array([bit_counts.get(label, 0) for label in ordered_labels])
probabilities = counts / counts.sum()

for label, count, prob in zip(ordered_labels, counts, probabilities, strict=True):
    print(f"P({label}) = {prob:.4f} ({count} shots)")

diagonal_probability = probabilities[0] + probabilities[3]
print(f"P(gg) + P(ee) = {diagonal_probability:.4f}")


In [ ]:
probability_matrix = np.array(
    [
        [bit_counts.get("00", 0), bit_counts.get("01", 0)],
        [bit_counts.get("10", 0), bit_counts.get("11", 0)],
    ],
    dtype=float,
)
probability_matrix /= probability_matrix.sum()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(ordered_labels, counts, color=["#4c78a8", "#f58518", "#e45756", "#54a24b"])
axes[0].set_title(f"{target}: classified bitstrings")
axes[0].set_xlabel("bitstring (0=g, 1=e)")
axes[0].set_ylabel("shots")

im = axes[1].imshow(probability_matrix, vmin=0.0, vmax=probability_matrix.max())
axes[1].set_title("First vs second measurement")
axes[1].set_xticks([0, 1], labels=["2nd: g", "2nd: e"])
axes[1].set_yticks([0, 1], labels=["1st: g", "1st: e"])

for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{probability_matrix[i, j]:.3f}", ha="center", va="center", color="white")

fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="probability")
fig.tight_layout()


If the calibration and waiting time are appropriate, the dominant populations should remain on the diagonal (`gg` and `ee`). Off-diagonal weight (`ge` or `eg`) indicates assignment error, relaxation during the wait, or insufficient classifier calibration.
